# FashionMNIST local training

Train a small PyTorch model on FashionMNIST locally (CPU or GPU).

In [1]:
import gzip
import shutil
import subprocess
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

## Download the dataset

Run the next two cells before training. They clone [fashion-mnist](https://github.com/zalandoresearch/fashion-mnist/blob/b2617bb6d3ffa2e429640350f613e3291e10b141/data/fashion/t10k-images-idx3-ubyte.gz) with git at commit `b2617bb6d3ffa2e429640350f613e3291e10b141`, print each check, and fail if the files are missing or incomplete.

In [2]:
DATA_ROOT = "/tmp/data"
RAW_DIR = Path(DATA_ROOT) / "FashionMNIST" / "raw"
CLONE_DIR = Path(DATA_ROOT) / "fashion-mnist-git"
EXPECTED_TRAIN_SAMPLES = 60000
GIT_BLOB_URL = (
    "https://github.com/zalandoresearch/fashion-mnist/blob/"
    "b2617bb6d3ffa2e429640350f613e3291e10b141/data/fashion/t10k-images-idx3-ubyte.gz"
)
GIT_REPO = "https://github.com/zalandoresearch/fashion-mnist.git"
GIT_COMMIT = "b2617bb6d3ffa2e429640350f613e3291e10b141"
GIT_DATA_PATH = "data/fashion"
GIT_FILE = "t10k-images-idx3-ubyte.gz"
RESOURCES = [
    "train-images-idx3-ubyte",
    "train-labels-idx1-ubyte",
    "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte",
]


def _run_git(args, cwd=None):
    cmd = ["git", *args]
    print(f"[git] {' '.join(cmd)}")
    subprocess.run(cmd, cwd=cwd, check=True)


def _extract_gz(src, dest):
    print(f"[download] extract {src} -> {dest}")
    with gzip.open(src, "rb") as f_in, dest.open("wb") as f_out:
        shutil.copyfileobj(f_in, f_out)


def download_dataset():
    print(f"[git] resource={GIT_BLOB_URL}")
    print(f"[git] repo={GIT_REPO}")
    print(f"[git] commit={GIT_COMMIT}")
    print(f"[git] clone dir={CLONE_DIR}")
    print(f"[download] raw dir={RAW_DIR}")
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)

    git_file = CLONE_DIR / GIT_DATA_PATH / GIT_FILE
    if (CLONE_DIR / ".git").is_dir():
        print(f"[git] repo already cloned, fetching {GIT_COMMIT}")
        _run_git(["sparse-checkout", "set", GIT_DATA_PATH], cwd=CLONE_DIR)
        _run_git(["fetch", "--depth", "1", "origin", GIT_COMMIT], cwd=CLONE_DIR)
        _run_git(["checkout", "--detach", GIT_COMMIT], cwd=CLONE_DIR)
    else:
        if CLONE_DIR.exists():
            shutil.rmtree(CLONE_DIR)
        print("[git] cloning with sparse checkout")
        _run_git(
            [
                "clone",
                "--filter=blob:none",
                "--sparse",
                "--no-checkout",
                GIT_REPO,
                str(CLONE_DIR),
            ]
        )
        _run_git(["sparse-checkout", "set", GIT_DATA_PATH], cwd=CLONE_DIR)
        _run_git(["fetch", "--depth", "1", "origin", GIT_COMMIT], cwd=CLONE_DIR)
        _run_git(["checkout", "--detach", GIT_COMMIT], cwd=CLONE_DIR)

    if not git_file.is_file():
        raise FileNotFoundError(f"[check] git did not download {git_file}")
    print(f"[check] {GIT_FILE}: ok ({git_file.stat().st_size} bytes) from git")

    git_dir = CLONE_DIR / GIT_DATA_PATH
    archives = sorted(git_dir.glob("*.gz"))
    if not archives:
        raise FileNotFoundError(f"[check] no .gz files in {git_dir}")
    for archive in archives:
        extracted = RAW_DIR / archive.name.removesuffix(".gz")
        _extract_gz(archive, extracted)

    missing_after = [name for name in RESOURCES if not (RAW_DIR / name).is_file()]
    if missing_after:
        raise FileNotFoundError(
            f"[check] incomplete extract in {RAW_DIR}: missing {missing_after}"
        )
    for name in RESOURCES:
        path = RAW_DIR / name
        print(f"[check] {name}: ok ({path.stat().st_size} bytes)")

    dataset = datasets.FashionMNIST(root=DATA_ROOT, train=True, download=False)
    print(f"[check] train samples={len(dataset)} expected={EXPECTED_TRAIN_SAMPLES}")
    if len(dataset) != EXPECTED_TRAIN_SAMPLES:
        raise RuntimeError(
            f"[check] expected {EXPECTED_TRAIN_SAMPLES} train samples, got {len(dataset)}"
        )
    print("[download] dataset is ready")
    return dataset

In [3]:
download_dataset()

[git] resource=https://github.com/zalandoresearch/fashion-mnist/blob/b2617bb6d3ffa2e429640350f613e3291e10b141/data/fashion/t10k-images-idx3-ubyte.gz
[git] repo=https://github.com/zalandoresearch/fashion-mnist.git
[git] commit=b2617bb6d3ffa2e429640350f613e3291e10b141
[git] clone dir=/tmp/data/fashion-mnist-git
[download] raw dir=/tmp/data/FashionMNIST/raw
[git] cloning with sparse checkout
[git] git clone --filter=blob:none --sparse --no-checkout https://github.com/zalandoresearch/fashion-mnist.git /tmp/data/fashion-mnist-git


Cloning into '/tmp/data/fashion-mnist-git'...


[git] git sparse-checkout set data/fashion
[git] git fetch --depth 1 origin b2617bb6d3ffa2e429640350f613e3291e10b141


From https://github.com/zalandoresearch/fashion-mnist
 * branch            b2617bb6d3ffa2e429640350f613e3291e10b141 -> FETCH_HEAD


[git] git checkout --detach b2617bb6d3ffa2e429640350f613e3291e10b141
[check] t10k-images-idx3-ubyte.gz: ok (4422102 bytes) from git
[download] extract /tmp/data/fashion-mnist-git/data/fashion/t10k-images-idx3-ubyte.gz -> /tmp/data/FashionMNIST/raw/t10k-images-idx3-ubyte
[download] extract /tmp/data/fashion-mnist-git/data/fashion/t10k-labels-idx1-ubyte.gz -> /tmp/data/FashionMNIST/raw/t10k-labels-idx1-ubyte
[download] extract /tmp/data/fashion-mnist-git/data/fashion/train-images-idx3-ubyte.gz -> /tmp/data/FashionMNIST/raw/train-images-idx3-ubyte
[download] extract /tmp/data/fashion-mnist-git/data/fashion/train-labels-idx1-ubyte.gz -> /tmp/data/FashionMNIST/raw/train-labels-idx1-ubyte
[check] train-images-idx3-ubyte: ok (47040016 bytes)
[check] train-labels-idx1-ubyte: ok (60008 bytes)
[check] t10k-images-idx3-ubyte: ok (7840016 bytes)
[check] t10k-labels-idx1-ubyte: ok (10008 bytes)
[check] train samples=60000 expected=60000
[download] dataset is ready


HEAD is now at b2617bb Merge pull request #175 from mikayelh/master


Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: /tmp/data
    Split: Train

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, inputs):
        inputs = self.flatten(inputs)
        logits = self.linear_relu_stack(inputs)
        return logits

In [5]:
def get_dataset():
    missing = [name for name in RESOURCES if not (RAW_DIR / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f"FashionMNIST is not downloaded (missing {missing}). Run download_dataset() first."
        )
    raw = datasets.FashionMNIST(
        root=DATA_ROOT,
        train=True,
        download=False,
    )
    images = raw.data.unsqueeze(1).float() / 255.0
    return TensorDataset(images, raw.targets)

In [6]:
def train():
    num_epochs = 3
    batch_size = 64

    dataset = get_dataset()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = NeuralNetwork().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

    for epoch in range(num_epochs):
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            pred = model(inputs)
            loss = criterion(pred, labels)
            loss.backward()
            optimizer.step()
        print(f"epoch: {epoch}, loss: {loss.item()}")

In [7]:
train()

epoch: 0, loss: 0.5474455952644348
epoch: 1, loss: 1.0071287155151367
epoch: 2, loss: 0.6750540733337402
